In [1]:
# Instalación
# Instalar dependencias
# ! pip install librosa
# ! pip install audioread soundfile   # backends de audio

# En Windows: descargar ffmpeg desde https://ffmpeg.org
# y añadirlo al PATH del sistema
import sys
!{sys.executable} -m pip install librosa audioread soundfile

# Script completo para el corpus
import librosa
import numpy as np
import os

KEY_NAMES = ['C','C#','D','D#','E','F','F#','G','G#','A','A#','B']
MAJOR_P = [6.35,2.23,3.48,2.33,4.38,4.09,2.52,5.19,2.39,3.66,2.29,2.88]
MINOR_P = [6.33,2.68,3.52,5.38,2.60,3.53,2.54,4.75,3.98,2.69,3.34,3.17]

def detect_key(y, sr):
    chroma = librosa.feature.chroma_cqt(y=y, sr=sr).mean(axis=1)
    best_score, best_key = -1, ''
    for i in range(12):
        rot = np.roll(chroma, -i)
        for profile, mode in [(MAJOR_P,'Major'),(MINOR_P,'Minor')]:
            score = np.corrcoef(rot, profile)[0,1]
            if score > best_score:
                best_score, best_key = score, f'{KEY_NAMES[i]} {mode}'
    return best_key

def analyze(path):
    y, sr = librosa.load(path, duration=90, mono=True)
    tempo, _ = librosa.beat.beat_track(y=y, sr=sr)
    key      = detect_key(y, sr)
    energy   = min(100, int(np.sqrt(np.mean(y**2)) * 1000))
    return round(float(tempo)), key, energy

folder = '../data/raw/Mp3-Top15-songs'
for f in sorted(os.listdir(folder)):
    if f.endswith('.mp3'):
        bpm, key, energy = analyze(os.path.join(folder, f))
        print(f'{f:<50} BPM:{bpm:>4}  Key:{key:<12}  Energy:{energy:>3}/100')


TypeError: only 0-dimensional arrays can be converted to Python scalars

## Comparación de progresiones — Normalización a Do

Transpone todas las progresiones a la misma tonalidad (C/Do) para poder comparar
si canciones distintas usan los mismos movimientos armónicos.

**3 visualizaciones:**
- 📋 Tabla agrupada por progresión compartida
- 🌡️ Heatmap de similitud entre canciones
- 🔗 Grafo de conexiones (requiere )


In [ ]:
# ══════════════════════════════════════════════════════════════
# CELDA — Comparación de progresiones normalizadas a Do (C)
# ══════════════════════════════════════════════════════════════
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import networkx as nx  # pip install networkx

# ─────────────────────────────────────────────────────────────
# PASO 1: Definir las notas en orden cromático
# ─────────────────────────────────────────────────────────────
# Cada nota tiene un número (0=C, 1=C#, 2=D ... 11=B)
# Esto nos permite hacer matemáticas con las notas

NOTE_TO_NUM = {
    'C': 0, 'C#': 1, 'D': 2, 'D#': 3, 'E': 4,  'F': 5,
    'F#': 6,'G': 7,  'G#': 8,'A': 9,  'A#': 10, 'B': 11
}
NUM_TO_NOTE = {v: k for k, v in NOTE_TO_NUM.items()}

# Nombres de los grados romanos (I, II, III...) para Mayor y menor
# Así el usuario ve "I → V → VIm → IV" en vez de "A → E → F#m → D"
DEGREE_NAMES_MAJOR = {0:'I', 2:'II', 4:'III', 5:'IV', 7:'V', 9:'VI', 11:'VII'}
DEGREE_NAMES_MINOR = {0:'Im',2:'IIm',3:'IIIb',5:'IVm',7:'Vm', 8:'VIb',10:'VIIb'}


# ─────────────────────────────────────────────────────────────
# PASO 2: Función para extraer la nota raíz de un acorde
# ─────────────────────────────────────────────────────────────
def chord_root(chord):
    """
    Dado un acorde como 'F#m' o 'D#', devuelve su nota raíz.
    'F#m' → 'F#'  (nota) y True (es menor)
    'D'   → 'D'   (nota) y False (es mayor)
    """
    is_minor = chord.endswith('m')
    root = chord.rstrip('m')   # quitar la 'm' del final si existe
    return root, is_minor


# ─────────────────────────────────────────────────────────────
# PASO 3: Transponer un acorde a Do (C)
# ─────────────────────────────────────────────────────────────
def transpose_chord_to_C(chord, key_root):
    """
    Transpone un acorde para que suene como si la tonalidad fuese C.
    
    Ejemplo:
      chord='E', key_root='A'  →  el E en tonalidad A es el V grado
      En tonalidad C, el V grado es G  →  devuelve 'G'
    
    key_root: nota raíz de la tonalidad (ej: 'A' si la key es 'A Major')
    """
    root, is_minor = chord_root(chord)
    
    if root not in NOTE_TO_NUM:
        return chord  # si no reconocemos la nota, la dejamos igual
    
    # Número de la nota del acorde (ej: E=4)
    chord_num = NOTE_TO_NUM[root]
    
    # Número de la tónica de la canción (ej: A=9)
    key_num   = NOTE_TO_NUM.get(key_root, 0)
    
    # Intervalo entre la tónica y el acorde (cuántos semitonos hay)
    # El % 12 hace que si pasamos de B volvamos a C (circular)
    interval  = (chord_num - key_num) % 12
    
    # Aplicar ese mismo intervalo desde C (que es 0)
    new_num   = interval % 12
    new_root  = NUM_TO_NOTE[new_num]
    
    return new_root + ('m' if is_minor else '')


# ─────────────────────────────────────────────────────────────
# PASO 4: Convertir acorde a grado romano
# ─────────────────────────────────────────────────────────────
def chord_to_degree(chord):
    """
    Convierte un acorde normalizado (ya en C) a su grado romano.
    'G' → 'V'   'Am' → 'VIm'   'F' → 'IV'
    """
    root, is_minor = chord_root(chord)
    num = NOTE_TO_NUM.get(root, -1)
    
    if is_minor:
        return DEGREE_NAMES_MINOR.get(num, f'?{num}m')
    else:
        return DEGREE_NAMES_MAJOR.get(num, f'?{num}')


# ─────────────────────────────────────────────────────────────
# PASO 5: Detectar tipo de movimiento armónico (3ª, 5ª, etc.)
# ─────────────────────────────────────────────────────────────
def movement_type(chord_a, chord_b):
    """
    Calcula el intervalo entre dos acordes y lo nombra.
    Útil para ver si la progresión usa movimientos de 3ª, 4ª, 5ª, etc.
    
    Devuelve: '5ª', '4ª', '3ª mayor', '3ª menor', '2ª', 'uníson', 'otro'
    """
    root_a, _ = chord_root(chord_a)
    root_b, _ = chord_root(chord_b)
    
    if root_a not in NOTE_TO_NUM or root_b not in NOTE_TO_NUM:
        return 'otro'
    
    # Intervalo en semitonos (el más corto de los dos caminos)
    diff = abs(NOTE_TO_NUM[root_b] - NOTE_TO_NUM[root_a])
    diff = min(diff, 12 - diff)  # siempre el intervalo más corto
    
    intervals = {
        0: 'uníson',
        1: '2ª menor', 2: '2ª mayor',
        3: '3ª menor', 4: '3ª mayor',
        5: '4ª justa', 6: 'tritono',
        7: '5ª justa',
    }
    return intervals.get(diff, 'otro')

# Colores para cada tipo de movimiento (para la visualización)
MOVEMENT_COLORS = {
    '5ª justa':  '#2ECC71',   # verde — movimiento más estable
    '4ª justa':  '#3498DB',   # azul
    '3ª mayor':  '#E67E22',   # naranja
    '3ª menor':  '#E74C3C',   # rojo
    '2ª mayor':  '#9B59B6',   # morado
    '2ª menor':  '#1ABC9C',   # turquesa
    'tritono':   '#F39C12',   # amarillo
    'uníson':    '#95A5A6',   # gris
    'otro':      '#BDC3C7',   # gris claro
}


# ─────────────────────────────────────────────────────────────
# PASO 6: Procesar todas las canciones
# ─────────────────────────────────────────────────────────────
# Datos del análisis previo (celda 3) — sustituye por df_meta si ya lo tienes
# Formato: (nombre_cancion, key_detectada, [lista_de_acordes])

SONGS_DATA = [
    ("505 - Arctic Monkeys",              "E Minor",  ["Em", "D",  "A",  "G"]),
    ("BTS - Butter",                      "G# Major", ["G#", "D#", "Cm", "F"]),
    ("BTS - Dynamite",                    "E Major",  ["E",  "B",  "C#m","A"]),
    ("BTS - FAKE LOVE",                   "D Minor",  ["Dm", "A#", "F",  "C"]),
    ("Radiohead - Creep (Live)",          "G Major",  ["G",  "B",  "C",  "Cm"]),
    ("Chappell Roan - Good Luck, Babe!",  "D Major",  ["D",  "A",  "Bm", "G"]),
    ("Coldplay x BTS - My Universe",      "E Major",  ["E",  "C#m","A",  "B"]),
    ("Mitski - My Love Mine All Mine",    "A Major",  ["A",  "E",  "F#m","D"]),
    ("Radiohead - No Surprises",          "F Major",  ["F",  "A#", "Am", "C"]),
    ("Sabrina Carpenter - Espresso",      "A Major",  ["A",  "E",  "F#m","D"]),
    ("TV Girl - Lovers Rock",             "D# Major", ["D#", "A#", "Gm", "F"]),
    ("Taylor Swift - Cruel Summer",       "A Major",  ["A",  "E",  "F#m","D"]),
    ("The Killers - Mr. Brightside",      "C# Major", ["C#", "G#", "A#m","F#"]),
    ("The Neighbourhood - Sweater Weather","G Minor", ["Gm", "D#", "A#", "F"]),
    ("Tyler - See You Again",             "C# Major", ["C#m","A",  "E",  "B"]),
]

# Procesar: normalizar a C y calcular grados
processed = []
for name, key, chords in SONGS_DATA:
    # Extraer la nota raíz de la key (ej: "A Major" → "A")
    key_root = key.split()[0]
    key_mode = key.split()[1]  # "Major" o "Minor"
    
    # Transponer cada acorde a C
    chords_in_C = [transpose_chord_to_C(c, key_root) for c in chords]
    
    # Convertir a grados romanos
    degrees = [chord_to_degree(c) for c in chords_in_C]
    
    # Calcular movimientos entre acordes consecutivos
    movements = [movement_type(chords[i], chords[i+1])
                 for i in range(len(chords)-1)]
    
    # Progresión como string para comparar fácilmente
    progression_str = ' → '.join(degrees)
    
    processed.append({
        'song':           name,
        'key':            key,
        'original':       ' → '.join(chords),
        'normalized':     ' → '.join(chords_in_C),
        'degrees':        progression_str,
        'movements':      movements,
        'chords_in_C':    chords_in_C,
    })

df_prog = pd.DataFrame(processed)
print("Progresiones normalizadas a C:\n")
print(df_prog[['song','key','original','degrees']].to_string(index=False))


# ─────────────────────────────────────────────────────────────
# VISUALIZACIÓN 1: Tabla agrupada por progresión compartida
# ─────────────────────────────────────────────────────────────
print("\n\n══ CANCIONES CON LA MISMA PROGRESIÓN ══\n")

# Agrupar canciones que tienen exactamente la misma progresión de grados
groups = df_prog.groupby('degrees')['song'].apply(list).reset_index()
groups.columns = ['Progresión (grados)', 'Canciones']
groups['Nº canciones'] = groups['Canciones'].apply(len)
groups = groups.sort_values('Nº canciones', ascending=False)

for _, row in groups.iterrows():
    if row['Nº canciones'] > 1:
        print(f"🎵 {row['Progresión (grados)']}")
        for song in row['Canciones']:
            print(f"   • {song}")
        print()


# ─────────────────────────────────────────────────────────────
# VISUALIZACIÓN 2: Heatmap de similitud entre canciones
# ─────────────────────────────────────────────────────────────
def chord_similarity(chords_a, chords_b):
    """
    Calcula cuántos acordes comparten dos canciones (normalizados a C).
    Devuelve un número entre 0 (nada en común) y 1 (exactamente iguales).
    """
    set_a = set(chords_a)
    set_b = set(chords_b)
    
    # Intersección dividida por unión (Jaccard similarity)
    intersection = len(set_a & set_b)
    union        = len(set_a | set_b)
    
    return intersection / union if union > 0 else 0

# Construir la matriz de similitud (canciones × canciones)
songs     = df_prog['song'].tolist()
n         = len(songs)
sim_matrix = np.zeros((n, n))

for i in range(n):
    for j in range(n):
        sim_matrix[i][j] = chord_similarity(
            df_prog.iloc[i]['chords_in_C'],
            df_prog.iloc[j]['chords_in_C']
        )

# Nombres cortos para el heatmap (sin artista)
short_names = [s.split(' - ')[-1][:20] for s in songs]

fig, ax = plt.subplots(figsize=(12, 10))
im = ax.imshow(sim_matrix, cmap='YlOrRd', vmin=0, vmax=1)

# Etiquetas de los ejes
ax.set_xticks(range(n))
ax.set_yticks(range(n))
ax.set_xticklabels(short_names, rotation=45, ha='right', fontsize=8)
ax.set_yticklabels(short_names, fontsize=8)

# Añadir el valor numérico en cada celda
for i in range(n):
    for j in range(n):
        val = sim_matrix[i][j]
        color = 'white' if val > 0.6 else 'black'
        ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                fontsize=7, color=color)

plt.colorbar(im, ax=ax, label='Similitud (0=nada en común, 1=idénticas)')
ax.set_title('🎵 Similitud de progresiones de acordes entre canciones\n(normalizadas a Do)', 
             fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()


# ─────────────────────────────────────────────────────────────
# VISUALIZACIÓN 3: Red/grafo de conexiones
# ─────────────────────────────────────────────────────────────
# Umbral: solo conectar canciones con similitud > 0.5
SIMILARITY_THRESHOLD = 0.5

G = nx.Graph()

# Añadir nodos (cada canción es un nodo)
for song in songs:
    G.add_node(song)

# Añadir aristas (conexiones entre canciones similares)
for i in range(n):
    for j in range(i+1, n):   # i+1 para no repetir pares
        sim = sim_matrix[i][j]
        if sim >= SIMILARITY_THRESHOLD:
            G.add_edge(songs[i], songs[j], weight=sim)

# Layout del grafo — spring_layout agrupa los nodos más conectados
pos = nx.spring_layout(G, seed=42, k=2.5)

fig, ax = plt.subplots(figsize=(14, 10))

# Dibujar aristas — más gruesas cuanto más similares
edges      = G.edges(data=True)
edge_weights = [d['weight'] * 4 for _, _, d in edges]
nx.draw_networkx_edges(G, pos, ax=ax,
                       width=edge_weights,
                       edge_color='#AED6F1',
                       alpha=0.7)

# Dibujar nodos
nx.draw_networkx_nodes(G, pos, ax=ax,
                       node_color='#2E86C1',
                       node_size=800,
                       alpha=0.9)

# Etiquetas de los nodos (nombres cortos)
labels = {s: s.split(' - ')[-1][:15] for s in songs}
nx.draw_networkx_labels(G, pos, labels=labels, ax=ax,
                        font_size=8, font_color='white',
                        font_weight='bold')

# Etiquetas en las aristas (valor de similitud)
edge_labels = {(u, v): f"{d['weight']:.2f}"
               for u, v, d in G.edges(data=True)}
nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels,
                              ax=ax, font_size=7, font_color='#1A5276')

ax.set_title(
    f'🔗 Red de canciones con progresiones similares\n'
    f'(umbral de similitud: {SIMILARITY_THRESHOLD})',
    fontsize=13, fontweight='bold', pad=15
)
ax.axis('off')
plt.tight_layout()
plt.show()

print(f"\nCanciones conectadas en el grafo: {G.number_of_edges()} conexiones")
print(f"Canciones aisladas (sin conexiones): "
      f"{sum(1 for n in G.nodes() if G.degree(n) == 0)}")
